# [4] 역대 지방선거 실시상황

**End Point**: `https://apis.data.go.kr/9760000/ScgnLocElctExctSttnService`  
**오퍼레이션**: `getScgnLocElctExctSttnInqire`

## 요청 파라미터 (PDF 확인)

| 파라미터 | 필수여부 | 설명 |
|---|---|---|
| serviceKey | 필수 | 인증키 |
| pageNo | 옵션 | 페이지 번호 |
| numOfRows | 옵션 | 목록 건수 |
| resultType | 옵션 | xml/json |

> ℹ️ `sgId` 등 선거 필터 파라미터 없음 — 전체 역대 데이터를 한 번에 조회합니다.

## 응답 컬럼 (PDF 확인)

| 컬럼명 | 한글명 | 필수 |
|---|---|---|
| num | 결과순서 | 필수 |
| elctNm | 선거명 | 필수 |
| elctNotmNm | 대별(회차명) | 옵션 |
| elctYmd | 선거일자 | 옵션 |
| elctDywk | 선거요일 | 옵션 |
| fdno | 정수 | 옵션 |
| voteRt | 투표율(%) | 옵션 |
| rmrk | 비고 | 옵션 |

**제공 범위**: 제3회 ~ 제8회 전국동시지방선거 (totalCount ≈ 21건)

In [5]:
# ✏️ 본인 Decoding 키로 교체하세요
API_KEY = "6lVhhlLRaGq/+tidZgS0POWCcl7BOqRJXiDj+Xtl/+rJJVEqNPWFjwFpyWkOn3NaNqacOHvj9UG+BnHAGBFd4w=="

In [6]:
import requests
import pandas as pd
import time

def fetch_all_pages(base_url, fixed_params, page_size=100, delay=0.3):
    all_items = []
    page = 1
    while True:
        params = {**fixed_params, "pageNo": str(page), "numOfRows": str(page_size), "resultType": "json"}
        try:
            resp = requests.get(base_url, params=params, timeout=15)
            if resp.status_code != 200:
                print(f"  ⚠️  HTTP {resp.status_code}: {resp.text[:300]}")
                break
            data = resp.json()
            header = data["response"]["header"]
            if header["resultCode"] not in ("INFO-00", "00"):
                print(f"  ⚠️  API 오류: {header['resultMsg']}")
                break
            body  = data["response"]["body"]
            total = int(body.get("totalCount", 0))
            raw   = body.get("items") or {}
            items = raw.get("item", []) if isinstance(raw, dict) else []
            if isinstance(items, dict):
                items = [items]
            all_items.extend(items)
            print(f"  페이지 {page}: {len(items)}건  (누적 {len(all_items)}/{total})")
            if len(all_items) >= total or not items:
                break
            page += 1
            time.sleep(delay)
        except Exception as e:
            print(f"  ❌ 오류 (페이지 {page}): {e}")
            break
    return all_items

def save_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"\n💾 저장 완료: {path}  ({len(df)}행 × {len(df.columns)}열)")

In [7]:
BASE_URL = "https://apis.data.go.kr/9760000/ScgnLocElctExctSttnService/getScgnLocElctExctSttnInqire"

# sgId 등 필터 파라미터 없이 전체 조회
print("📌 역대 지방선거 실시상황 수집 중...")
items = fetch_all_pages(
    base_url=BASE_URL,
    fixed_params={"serviceKey": API_KEY},
    page_size=100,
)
print(f"\n✅ 총 {len(items)}건 수집 완료")

📌 역대 지방선거 실시상황 수집 중...
  페이지 1: 19건  (누적 19/19)

✅ 총 19건 수집 완료


In [8]:
if not items:
    print("⚠️  데이터 없음")
else:
    df = pd.DataFrame(items)

    # PDF 확인된 실제 컬럼명
    col_map = {
        "num":        "결과순서",
        "elctNm":     "선거명",
        "elctNotmNm": "대별(회차명)",
        "elctYmd":    "선거일자",
        "elctDywk":   "선거요일",
        "fdno":       "정수",
        "voteRt":     "투표율(%)",
        "rmrk":       "비고",
    }
    existing = [c for c in col_map if c in df.columns]
    df = df[existing].rename(columns=col_map)

    # 숫자형 변환
    for col in ["정수", "투표율(%)"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    display(df)
    save_csv(df, "04_역대지방선거실시상황.csv")

,결과순서,선거명,대별(회차명),선거일자,선거요일,정수,투표율(%),비고
0,1,전국동시지방선거,제7회,20180613,수,4016,60.2,
1,2,전국동시지방선거,제8회,20220601,수,4125,50.9,
2,3,시·읍·면의회의원선거,,19520425,금,17559,90.7,
3,4,시·도의회의원선거,,19520510,토,306,81.2,
4,5,시·읍·면장선거,,19560808,수,582,86.7,
5,6,시·읍·면의회의원선거,,19560808,수,16961,79.6,
6,7,시·도의회의원선거,,19560813,월,437,85.8,
7,8,시·도의회의원선거,,19601212,월,487,67.4,
8,9,시·읍·면의회의원선거,,19601219,월,16909,78.9,
9,10,시·읍·면장선거,,19601226,월,1468,75.4,



💾 저장 완료: 04_역대지방선거실시상황.csv  (19행 × 8열)
